# Training the leaf-disease CNN (MobileNetV2, potato)

This notebook trains the model that looks at a photo of a potato leaf and says whether it shows **early blight**, **late blight**, or is **healthy**. It runs top to bottom in Google Colab (*Runtime → Run all*) on a GPU runtime. The only optional edit is `MOUNT_DRIVE = True` in the first code cell, to keep the outputs in Google Drive.

**What each step does, in plain language**

1. **Data.** The images come from PlantVillage, a public collection of leaf photos taken against a plain background (TensorFlow Datasets `plant_village`, CC0 licence). Only the three potato classes are kept. Their exact names (`Potato___Early_blight`, `Potato___Late_blight`, `Potato___healthy`) are read from the dataset's own metadata and checked, rather than typed in, so a change in the dataset stops the notebook instead of silently mislabelling images. The class counts are printed: 1,000 early blight, 1,000 late blight and only 152 healthy.

2. **The same split as the rest of the project.** The images are divided 70 / 15 / 15 into training, validation and test sets, stratified by class with a fixed random seed (42). The notebook rebuilds this split and **checks it is identical**, file for file, to `data/splits/train.csv`, `val.csv` and `test.csv` in the project repository. So the test set here is exactly the one used locally, and the test images are never seen during training.

3. **Preparing the images.** Every image is resized to 224 × 224 pixels and scaled the way MobileNetV2 expects (`preprocess_input`, values from −1 to 1). Training images are also randomly rotated, flipped, zoomed and brightened a little each time they are shown. This *augmentation* teaches the model that a leaf is still the same leaf at a slightly different angle or in different light. It is applied to training images only, so validation and test images are judged as they are.

4. **The model.** MobileNetV2 is a small, fast network already trained on 1.4 million everyday photos (ImageNet). Its early layers already detect edges, textures and shapes. We keep that knowledge (*transfer learning*) and add a small new "head": average pooling, batch normalisation, dropout, a 128-unit layer, more dropout, and a final 3-way softmax that outputs one probability per class.

5. **Phase 1: train the head only.** The pretrained network is frozen and only the new head learns (Adam optimiser, learning rate 0.001, up to 15 passes over the data). Training stops early if the validation loss has not improved for 3 passes, and the best weights are kept. Because healthy leaves are much rarer, mistakes on them are weighted more heavily (*class weights* computed from the training counts), so the model cannot do well simply by ignoring the healthy class.

6. **Phase 2: fine-tune.** The upper part of MobileNetV2 (layer 100 onwards) is unfrozen and trained very gently (learning rate 0.00001, up to 15 passes, the same early stopping) to adapt it to potato leaves. Its batch-normalisation layers stay in inference mode, so their statistics learned on ImageNet are not disturbed by our small batches.

7. **Evaluation on the held-out test set.** The 324 test images are classified once, at the end. We report overall accuracy and, per class, precision (when the model says "late blight", how often it is right), recall (of the real late-blight leaves, how many it finds) and F1 (the balance of the two). Because the classes are imbalanced, macro-averaged F1 is the headline number, not accuracy. A confusion matrix shows which classes get mixed up, and the training curves show how loss and accuracy evolved in both phases.

8. **Export.** The model is saved in Keras format and converted to TensorFlow Lite (with the default size optimisation), which is what the backend server runs. `labels.txt` lists the class names in the order of the model's outputs. To check the conversion did not change the model's behaviour, the TFLite and Keras versions classify the same 50 test images and the agreement is printed.

9. **Download.** The last cell zips the TFLite model, `labels.txt`, the metrics (JSON) and the two figures for download.

*Caveat for the write-up:* PlantVillage photos are single leaves on plain backgrounds in controlled light. Accuracy here is an upper bound, not an estimate of performance in the field. `ml/eval_cnn.py` measures that lab-to-field gap on the farmer's own photos.

## 0. Settings and environment

In [ ]:
import os

# The only optional edit: True keeps every output in Google Drive (MyDrive/blight-ews/cnn_run).
MOUNT_DRIVE = False

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
PHASE1_EPOCHS = 15
PHASE2_EPOCHS = 15
PATIENCE = 3              # early stopping: passes without val_loss improvement
UNFREEZE_FROM = 100       # phase 2 trains MobileNetV2 layers from this index onwards
TFLITE_CHECK_IMAGES = 50

# The committed split files (public repository), and the PlantVillage archive that TFDS uses.
REPO_RAW = "https://raw.githubusercontent.com/crappyrizz/blight-ews/main"
ARCHIVE_URL = ("https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/"
               "d5652a28-c1d8-4b76-97f3-72fb80f94efc/file_downloaded")

# For checking the notebook on a laptop only; never set in Colab.
SMOKE_TEST = os.environ.get("CNN_SMOKE_TEST") == "1"
if SMOKE_TEST:
    PHASE1_EPOCHS = PHASE2_EPOCHS = 1

In [ ]:
import json
import subprocess
import sys
import urllib.request
import zipfile
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(SEED)  # Python, NumPy and TensorFlow seeds in one call
IN_COLAB = "google.colab" in sys.modules

print("TensorFlow", tf.__version__, "| Keras", tf.keras.__version__, "| TFDS", tfds.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE. In Colab: Runtime > Change runtime type > GPU, then Run all.")

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/blight-ews/cnn_run")
elif IN_COLAB:
    OUTPUT_DIR = Path("/content/cnn_outputs")
else:
    OUTPUT_DIR = Path(os.environ.get("CNN_OUTPUT_DIR", "cnn_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content") if IN_COLAB else OUTPUT_DIR
print("Outputs go to", OUTPUT_DIR.resolve())

## Helper functions: class names and the project's split

Pure Python (no TensorFlow), self-contained. These reproduce exactly what `ml/prepare_images.py` did when it built `data/splits/*.csv`. The project's test suite runs this cell to check that it still does.

In [ ]:
import csv
import io
import random

SPECIES = "potato"
# Class index order for the model's outputs and labels.txt (alphabetical).
CLASS_NAMES = ["early_blight", "healthy", "late_blight"]
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
EXPORT_DIR = "data/plantvillage_potato"  # how the split CSVs name the images


def short_class_name(dataset_label):
    """'Potato___Early_blight' -> 'early_blight'."""
    return dataset_label.partition("___")[2].lower()


def potato_label_names(all_label_names):
    """{dataset label name: short class name} for the potato classes, checked.

    Read from the dataset's own metadata. Stops if the potato classes are not
    exactly the three expected ones, instead of guessing.
    """
    found = {name: short_class_name(name) for name in all_label_names
             if name.split("___")[0].lower() == SPECIES}
    if sorted(found.values()) != CLASS_NAMES:
        raise ValueError(f"Expected potato classes {CLASS_NAMES}, the dataset has {sorted(found.values())}")
    return found


def export_paths(filenames_by_class):
    """{class: {source filename: split-CSV path}}.

    ml/prepare_images.py wrote each class's images as <class>_00000.jpg,
    <class>_00001.jpg, ... in sorted order of the original archive filenames
    (no image was dropped as unreadable or duplicate), so the same rule maps
    TFDS's 'image/filename' back to the names in the split CSVs.
    """
    return {
        class_name: {
            source: f"{EXPORT_DIR}/{class_name}/{class_name}_{index:05d}.jpg"
            for index, source in enumerate(sorted(filenames))
        }
        for class_name, filenames in filenames_by_class.items()
    }


def rebuild_split(filenames_by_class, seed=SEED):
    """The 70/15/15 stratified split, rebuilt exactly as ml/prepare_images.py builds it.

    Per class: sort, shuffle with random.Random(seed), first 70 % train, next
    15 % val, the remainder test. Returns {split: [(filepath, label), ...]} in
    the same row order as the CSV files.
    """
    paths = export_paths(filenames_by_class)
    splits = {name: [] for name in SPLIT_FRACTIONS}
    for class_name in sorted(paths):
        shuffled = sorted(paths[class_name].values())
        random.Random(seed).shuffle(shuffled)
        n_train = int(len(shuffled) * SPLIT_FRACTIONS["train"])
        n_val = int(len(shuffled) * SPLIT_FRACTIONS["val"])
        chunks = {"train": shuffled[:n_train],
                  "val": shuffled[n_train:n_train + n_val],
                  "test": shuffled[n_train + n_val:]}
        for split_name, chunk in chunks.items():
            splits[split_name].extend((path, class_name) for path in chunk)
    return splits


def parse_split_csv(text):
    """Rows of a split CSV (header: filepath,label) as [(filepath, label), ...]."""
    rows = list(csv.reader(io.StringIO(text)))
    assert rows[0] == ["filepath", "label"], f"unexpected header {rows[0]}"
    return [(path, label) for path, label in rows[1:] if path]

## 1. Data: PlantVillage, potato classes only

The class names are taken from the dataset's metadata. The images are loaded through TensorFlow Datasets. Mendeley, which hosts the archive, refuses some download clients (HTTP 403). If that happens, the same archive is fetched with `curl` and read with the same rule TFDS's own `plant_village` builder uses (the `Plant_leave_diseases_dataset_without_augmentation/<label>/*.jpg` files, their original filenames, decoded by TensorFlow), so the images, labels and filenames are identical either way.

In [ ]:
builder = tfds.builder("plant_village")
ALL_LABELS = builder.info.features["label"].names
POTATO = potato_label_names(ALL_LABELS)  # dataset label name -> short class name
print(f"plant_village has {len(ALL_LABELS)} classes. Potato classes found in its metadata:")
for dataset_name, short in sorted(POTATO.items(), key=lambda item: item[1]):
    print(f"  {dataset_name!r:28} -> {short}")


def load_with_tfds():
    builder.download_and_prepare()
    dataset = builder.as_dataset(split="train", shuffle_files=False)
    for example in tfds.as_numpy(dataset):
        label = ALL_LABELS[int(example["label"])]
        if label in POTATO:
            yield POTATO[label], example["image/filename"].decode(), example["image"]


def load_from_archive():
    archive = Path(os.environ.get("PLANTVILLAGE_ZIP", WORK_DIR / "plantvillage.zip"))
    if not archive.exists():
        print("Downloading the PlantVillage archive with curl (about 870 MB)...")
        subprocess.run(["curl", "-sSL", "--fail", "-A", "Mozilla/5.0", "-o", str(archive), ARCHIVE_URL],
                       check=True)
    with zipfile.ZipFile(archive) as zf:
        for name in sorted(zf.namelist()):
            parts = name.split("/")
            if (len(parts) == 3 and parts[0] == "Plant_leave_diseases_dataset_without_augmentation"
                    and parts[1] in POTATO and parts[2].lower().endswith(".jpg")):
                image = tf.io.decode_jpeg(zf.read(name), channels=3).numpy()
                yield POTATO[parts[1]], parts[2], image


try:
    examples = list(load_with_tfds())
    DATA_SOURCE = "tensorflow_datasets plant_village"
except Exception as error:
    print(f"TFDS download failed ({type(error).__name__}: {str(error)[:200]})")
    print("Falling back to the same archive via curl.")
    examples = list(load_from_archive())
    DATA_SOURCE = "plant_village archive (same files as TFDS), via curl"

IMAGES = {(label, filename): image for label, filename, image in examples}
counts = Counter(label for label, _, _ in examples)
print(f"\nLoaded {len(IMAGES)} potato images from {DATA_SOURCE}")
for class_name in CLASS_NAMES:
    print(f"  {class_name:13} {counts[class_name]:5}")

## 2. The same train / val / test split as the project

The split is rebuilt here with seed 42 and compared row for row with the committed `data/splits/*.csv` (fetched from the project's public GitHub repository). If they differ in any way, the notebook stops.

In [ ]:
def committed_split(name):
    """The committed CSV: from GitHub, or uploaded to Colab / Drive if offline."""
    try:
        with urllib.request.urlopen(f"{REPO_RAW}/data/splits/{name}.csv", timeout=30) as response:
            return parse_split_csv(response.read().decode("utf-8"))
    except Exception as error:
        for folder in (Path("."), Path("/content"), Path("/content/drive/MyDrive/blight-ews/splits")):
            if (folder / f"{name}.csv").exists():
                return parse_split_csv((folder / f"{name}.csv").read_text(encoding="utf-8"))
        raise RuntimeError(f"Could not fetch {name}.csv ({error}). Upload data/splits/{name}.csv "
                           "to /content (Files panel) and run this cell again.") from error


filenames_by_class = {c: [f for (label, f) in IMAGES if label == c] for c in CLASS_NAMES}
rebuilt = rebuild_split(filenames_by_class)
for split_name in ("train", "val", "test"):
    committed = committed_split(split_name)
    assert rebuilt[split_name] == committed, f"{split_name} split differs from data/splits/{split_name}.csv"
    print(f"{split_name:5}: {len(committed):5} images, identical to data/splits/{split_name}.csv  "
          f"{dict(Counter(label for _, label in committed))}")

# Split-CSV path -> image, via the original filename.
by_path = {}
for class_name, mapping in export_paths(filenames_by_class).items():
    for source_filename, path in mapping.items():
        by_path[path] = IMAGES[(class_name, source_filename)]


def as_arrays(rows):
    images = np.stack([by_path[path] for path, _ in rows])
    labels = np.array([CLASS_NAMES.index(label) for _, label in rows], dtype=np.int32)
    return images, labels


if SMOKE_TEST:  # laptop check only: a few training images per class, after the split check
    def first_per_class(rows, n):
        return [row for c in CLASS_NAMES for row in [r for r in rows if r[1] == c][:n]]
    rebuilt["train"] = first_per_class(rebuilt["train"], 24)
    rebuilt["val"] = first_per_class(rebuilt["val"], 8)

x_train, y_train = as_arrays(rebuilt["train"])
x_val, y_val = as_arrays(rebuilt["val"])
x_test, y_test = as_arrays(rebuilt["test"])
TEST_PATHS = [path for path, _ in rebuilt["test"]]
del examples, IMAGES  # free memory; the arrays hold what is needed
print("Arrays:", x_train.shape, x_val.shape, x_test.shape)

## 3. Preprocessing and augmentation

Resize to 224 × 224 (bilinear, antialiased), then `mobilenet_v2.preprocess_input` (pixel values scaled from 0..255 to −1..1). Training images only are randomly rotated (±10 % of a full turn), flipped left-right, zoomed (±15 %) and brightened or darkened (±20 %).

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1, seed=SEED),
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomZoom(0.15, seed=SEED),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255), seed=SEED),
], name="augmentation")


def resize(images):
    return tf.image.resize(tf.cast(images, tf.float32), (IMAGE_SIZE, IMAGE_SIZE),
                           method="bilinear", antialias=True)


def make_dataset(images, labels, training):
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        dataset = dataset.shuffle(len(images), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(BATCH_SIZE).map(lambda x, y: (resize(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        dataset = dataset.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return dataset.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val, training=False)
test_ds = make_dataset(x_test, y_test, training=False)

# A look at what the model sees in training (augmented), undone back to 0..1 for display.
batch_images, batch_labels = next(iter(train_ds))
plt.figure(figsize=(10, 4))
for i in range(min(8, len(batch_images))):
    plt.subplot(2, 4, i + 1)
    plt.imshow((batch_images[i].numpy() + 1) / 2)
    plt.title(CLASS_NAMES[int(batch_labels[i])], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Model: MobileNetV2 + a small classification head

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
                                         include_top=False, weights="imagenet")
base.trainable = False

inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="image")
# training=False keeps MobileNetV2's batch-normalisation layers in inference mode,
# in phase 1 and still after they are unfrozen in phase 2.
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", name="probabilities")(x)
model = tf.keras.Model(inputs, outputs, name="potato_mobilenetv2")
model.summary(show_trainable=True)

## 5. Phase 1: train the head (base frozen)

Class weights are "balanced": each class's weight is `n_train / (3 × n_class)`, so the rare healthy class counts for more per image.

In [ ]:
weights = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y_train)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(weights)}
print("Training counts:", {CLASS_NAMES[i]: int((y_train == i).sum()) for i in range(len(CLASS_NAMES))})
print("Class weights:  ", {CLASS_NAMES[i]: round(w, 3) for i, w in CLASS_WEIGHT.items()})


def callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                             restore_best_weights=True, verbose=1)]


model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history1 = model.fit(train_ds, validation_data=val_ds, epochs=PHASE1_EPOCHS,
                     class_weight=CLASS_WEIGHT, callbacks=callbacks())

## 6. Phase 2: fine-tune MobileNetV2 from layer 100 onwards

Layers from index 100 are made trainable, except batch-normalisation layers, which stay frozen and (via `training=False` above) in inference mode. The learning rate is 100 times smaller, so the pretrained features are adjusted, not overwritten.

In [ ]:
base.trainable = True
for index, layer in enumerate(base.layers):
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = index >= UNFREEZE_FROM
print(f"MobileNetV2 has {len(base.layers)} layers; trainable now: "
      f"{sum(layer.trainable for layer in base.layers)} (from index {UNFREEZE_FROM}, BatchNorm excluded)")

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history2 = model.fit(train_ds, validation_data=val_ds, epochs=PHASE2_EPOCHS,
                     class_weight=CLASS_WEIGHT, callbacks=callbacks())

## 7. Evaluation on the held-out test set

In [ ]:
test_probabilities = model.predict(test_ds, verbose=0)
y_pred = test_probabilities.argmax(axis=1)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0)
report_dict = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
matrix = confusion_matrix(y_test, y_pred, labels=range(len(CLASS_NAMES)))

print(f"Test images: {len(y_test)}   accuracy: {accuracy:.3f}   macro F1: {macro_f1:.3f}\n")
print(report)

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix: PlantVillage test set")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

# Training curves, both phases joined, with the start of fine-tuning marked.
history = {key: history1.history[key] + history2.history[key]
           for key in ("loss", "val_loss", "accuracy", "val_accuracy")}
epochs = range(1, len(history["loss"]) + 1)
phase1_epochs_run = len(history1.history["loss"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in ((axes[0], "loss", "Loss"), (axes[1], "accuracy", "Accuracy")):
    ax.plot(epochs, history[metric], marker="o", label="train")
    ax.plot(epochs, history[f"val_{metric}"], marker="o", label="validation")
    ax.axvline(phase1_epochs_run + 0.5, color="grey", linestyle="--", label="fine-tuning starts")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

## 8. Export: Keras, TensorFlow Lite, labels, and a TFLite-vs-Keras check

The TFLite model expects a float32 batch of shape `[1, 224, 224, 3]`, already resized and passed through `preprocess_input` (RGB, values −1..1). It outputs 3 probabilities in the order of `labels.txt`.

In [ ]:
model.save(OUTPUT_DIR / "cnn.keras")

saved_model_dir = WORK_DIR / "cnn_saved_model"
model.export(str(saved_model_dir))  # SavedModel: the most reliable route to TFLite with Keras 3
converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()
(OUTPUT_DIR / "cnn.tflite").write_bytes(tflite_bytes)
(OUTPUT_DIR / "labels.txt").write_text("\n".join(CLASS_NAMES) + "\n", encoding="utf-8")
print(f"cnn.keras, cnn.tflite ({len(tflite_bytes) / 1e6:.1f} MB) and labels.txt written to {OUTPUT_DIR}")

# Does the TFLite model agree with the Keras model?
interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]

n_check = min(TFLITE_CHECK_IMAGES, len(x_test))
check_inputs = preprocess_input(resize(x_test[:n_check])).numpy()
keras_probabilities = model.predict(check_inputs, verbose=0)
tflite_probabilities = []
for image in check_inputs:
    interpreter.set_tensor(input_detail["index"], image[np.newaxis].astype(input_detail["dtype"]))
    interpreter.invoke()
    tflite_probabilities.append(interpreter.get_tensor(output_detail["index"])[0])
tflite_probabilities = np.array(tflite_probabilities)

agreement = float((keras_probabilities.argmax(1) == tflite_probabilities.argmax(1)).mean())
max_difference = float(np.abs(keras_probabilities - tflite_probabilities).max())
print(f"TFLite vs Keras on {n_check} test images: {agreement:.1%} same predicted class, "
      f"largest probability difference {max_difference:.4f}")

## 9. Metrics file and download

In [ ]:
metrics = {
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "data_source": DATA_SOURCE,
    "classes": CLASS_NAMES,
    "split_sizes": {"train": len(y_train), "val": len(y_val), "test": len(y_test)},
    "split_matches_repo_csvs": True,
    "smoke_test": SMOKE_TEST,
    "class_weights": {CLASS_NAMES[i]: w for i, w in CLASS_WEIGHT.items()},
    "epochs_run": {"phase1": phase1_epochs_run, "phase2": len(history2.history["loss"])},
    "test": {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "per_class": {c: {k: report_dict[c][k] for k in ("precision", "recall", "f1-score", "support")}
                      for c in CLASS_NAMES},
        "confusion_matrix": {"rows_true_cols_predicted": CLASS_NAMES, "values": matrix.tolist()},
    },
    "tflite_check": {"images": n_check, "agreement": agreement, "max_probability_difference": max_difference,
                     "size_bytes": len(tflite_bytes)},
    "model_input": {"shape": [1, IMAGE_SIZE, IMAGE_SIZE, 3], "dtype": "float32",
                    "preprocessing": "RGB, bilinear antialiased resize to 224x224, then "
                                     "mobilenet_v2.preprocess_input (x / 127.5 - 1)"},
    "versions": {"tensorflow": tf.__version__, "tfds": tfds.__version__},
}
(OUTPUT_DIR / "cnn_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

bundle = OUTPUT_DIR / "cnn_outputs.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ("cnn.tflite", "labels.txt", "cnn_metrics.json", "confusion_matrix.png", "training_curves.png"):
        zf.write(OUTPUT_DIR / name, arcname=name)
print(f"{bundle} ({bundle.stat().st_size / 1e6:.1f} MB):", ", ".join(zipfile.ZipFile(bundle).namelist()))
print("In the project: cnn.tflite and labels.txt go in models/; the JSON and PNGs in docs/.")

if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))